### Here we will read required Dimension  csv

In [0]:
catalog_name = "ecommerce"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
import pyspark.sql.functions as F

schema_brand = StructType(
    [
        StructField("brand_code",StringType()),
        StructField("brand_name",StringType()),
        StructField("category_code", StringType())
    ]
)

df = spark.read.option("header", True).format("csv").schema(schema_brand).load("/Volumes/ecommerce/raw/raw_aws_data/brands/brands.csv")

In [0]:
df_brands = spark.read.option("header",True).format("csv").load("/Volumes/ecommerce/raw/raw_aws_data/brands/brands.csv")
df_brands = df_brands \
  .withColumn("_file_path", F.col("_metadata.file_path")) \
  .withColumn("Injested_at", F.current_timestamp()
)

df_brands.show(5,False)

### Write csv which was read as DF to table (delta)

In [0]:

df_brands.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", True) \
    .saveAsTable(f"{catalog_name}.bronze.brz_brands")
    

### lets write python to do same process for all tables to braonze.

In [0]:
%skip

volume_path =  "/Volumes/ecommerce/raw/raw_aws_data"
bronze_dwh_path = f"{catalog_name}.bronze"


def load_to_bronze(folder_name: str) -> None:
    df = spark \
        .read \
        .option("header", True) \
        .format("csv") \
        .load(f"{volume_path}/{folder_name}")
    
    df = df.withColumn("_file_path", F.col("_metadata.file_path")) \
        .withColumn("Injested_at", F.current_timestamp())

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", True) \
        .saveAsTable(f'{bronze_dwh_path}.brz_{folder_name}')

for each_folder in (dbutils.fs.ls("/Volumes/ecommerce/raw/raw_aws_data")):
    if each_folder.name.startswith("."):
        continue
    else:
        dir_name = each_folder.name.rstrip("/")
        print(f'Bronze load started for {dir_name}...')
        load_to_bronze(dir_name)
        print(f'Bronze load completed for {dir_name}...')



## Advanced python approach using DataClass


In [0]:
from dataclasses import dataclass
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, DoubleType, FloatType

@dataclass
class Csvconfig:
    foldername: str
    schema: StructType
    target_name: str
    

brand_config = Csvconfig(
    foldername = "brands",
    schema = StructType([
        StructField("brand_code",StringType()),
        StructField("brand_name",StringType()),
        StructField("category_code", StringType())
    ]
    ),
    target_name = "brz_brands"
   
)

category_config = Csvconfig(
    foldername= "category",
    schema = StructType([
        StructField("category_code", StringType()),
        StructField("category_name", StringType())
    ]
    ),
    target_name = "brz_category",
)

customer_config = Csvconfig(
    foldername = "customers",
    schema = StructType([
        StructField("customer_id", StringType()),
        StructField("phone", IntegerType()),
        StructField("country_code", StringType()),
        StructField("country", StringType()),
        StructField("state", StringType())
    ]
    ),
    target_name = "brz_customers"
)

date_config = Csvconfig(
    foldername = "date",
    schema = StructType([
        StructField("date", DateType()),
        StructField("year", IntegerType()),
        StructField("day_name", StringType()),
        StructField("quarter", IntegerType()),
        StructField("week_of_year", IntegerType())
    ]
    ),
    target_name = "brz_date"

)

orderItems_config = Csvconfig(
    foldername = "order_items",
    schema = StructType([
        StructField("dt", DateType(), True),                     # Date of the order
        StructField("order_ts", TimestampType(), True),          # Order timestamp
        StructField("customer_id", StringType(), True),          # Customer ID (string in case of alphanumeric IDs)
        StructField("order_id", IntegerType(), True),             # Order ID
        StructField("item_seq", IntegerType(), True),            # Item sequence number
        StructField("product_id", StringType(), True),           # Product ID
        StructField("quantity", StringType(), True),             # Quantity (could be fractional)
        StructField("unit_price_currency", StringType(), True),  # Currency code (USD, GBP, etc.)
        StructField("unit_price", DoubleType(), True),           # Unit price
        StructField("discount_pct", DoubleType(), True),         # Discount percentage
        StructField("tax_amount", DoubleType(), True),           # Tax amount
        StructField("channel", StringType(), True),              # Sales channel (online, store, etc.)
        StructField("coupon_code", StringType(), True)
    ]
    ),
    target_name = "brz_order_items"
)

products_config = Csvconfig(
    foldername = "products",
    schema = StructType([
        StructField("product_id", StringType(), False),
    StructField("sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(), True),  #datatype is string due to incoming data contain anamolies
    StructField("length_cm", StringType(), True),     #datatype is string due to incoming data contain anamolies
    StructField("width_cm", FloatType(), True),
    StructField("height_cm", FloatType(), True),
    StructField("rating_count", IntegerType(), True),
    StructField("file_name", StringType(), False),
    StructField("ingest_timestamp", TimestampType(), False)
    ]
    ),
    target_name = "brz_products"
)


# put all configs in list of configs. 

config_list = [brand_config, products_config, orderItems_config, date_config, customer_config, category_config]

### Now loop through each config and load data into bronze

In [0]:
brz_schema_name = "ecommerce.bronze"
def load_to_bronze(cfg: Csvconfig) -> None:

    # Read into DF 
    df = spark.read \
        .format("csv") \
        .option("header", True) \
        .schema(cfg.schema) \
        .load(f"/Volumes/ecommerce/raw/raw_aws_data/{cfg.foldername}")
    
    # Write DF to schema Bronze.
    """
    Now PAY GOOF ATTENTION HERE. 
    We can write this DF as 
    A. Paruet 
    B. Delta 
    A - > if we write it as Parquet it will be written as files based on location we provide or not provide 
    When location is proveded 
        - df.write.format.save("user procided location) = this could be volume / S3 / Azure etc. not schema as this is file format.
        - df.write.format.saveAsTable(<schema.path>) = in given schema but as this is manged table DBR will decide location of files.
    B -> - df.write.fomrat(delta).save("provided location / volume")  this could be volume / S3 / Azure etc. not schema as this is file format.
         - df.write.fomrat(delta).saveAsTable(<schema>) in given schema but as this is manged table DBR will decide location of files.
    Important thing to note here - both Parquet and delta are files in backend :)
    """
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema","true") \
        .saveAsTable(f"{brz_schema_name}.target_name")

for each_config in config_list:
    print(f"Writing for {each_config.foldername}")
    load_to_bronze(each_config)

  

#### MergeSchema Vs OverwriteSchem
#### A.  OverwriteSchema
  - if new column added / removed or data type has changed then we should use mode(overwrite) with overwriteSchema() as we are applying new schema 
#### B. MergeSceham 
  - If new column is added but there is no change in existing column in terms of data type of number of columns then use this with mode(append) , this will put null for old rows for newly added column
    
